# Early Insights and Opportunities from AI Usage and Developer Trend in Algeria

In [1]:
from pathlib import Path
from urllib.request import urlopen
import json
import ssl

import altair as alt
import pandas as pd
import pycountry
import attaviz
import squarify

attaviz.enable()


def find_project_root(marker="pyproject.toml"):
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
ANTHROPIC_DIR = DATA_DIR / "AI" / "Anthropic"
GITHUB_DIR = DATA_DIR / "github"

COUNTRY_CODE = "DZ"
COUNTRY_NAME = "Algeria"

PEER_GROUPS = {
    "Regional": ["EG", "JO", "TN", "IQ"],
    "Structural": ["EC", "PE", "GH", "VN", "CO"],
    "Aspirational": ["MY", "CL", "PL", "RO"],
}
COUNTRY_NAMES = {
    "DZ": "Algeria",
    "EG": "Egypt",
    "JO": "Jordan",
    "TN": "Tunisia",
    "IQ": "Iraq",
    "EC": "Ecuador",
    "PE": "Peru",
    "GH": "Ghana",
    "VN": "Vietnam",
    "CO": "Colombia",
    "MY": "Malaysia",
    "CL": "Chile",
    "PL": "Poland",
    "RO": "Romania",
    "NG": "Nigeria",
    "KE": "Kenya",
}
GROUP_OF = {c: g for g, members in PEER_GROUPS.items() for c in members}
GROUP_OF[COUNTRY_CODE] = "Algeria"
GROUP_ORDER = ["Algeria", "Regional", "Structural", "Aspirational"]
GROUP_COLORS = {
    "Algeria": "#0071BC",
    "Regional": "#8A969F",
    "Structural": "#8A969F",
    "Aspirational": "#8A969F",
}
PEER_CODES = [COUNTRY_CODE] + [c for g in PEER_GROUPS.values() for c in g]
ALGERIA_COLOR = "#0071BC"
PEER_COLORS = ["#8A969F", "#9FA8AF", "#B0B8BE", "#C1C8CD", "#D2D7DB"]

ISO2_TO_ISO3 = {
    "DZ": "DZA",
    "EG": "EGY",
    "JO": "JOR",
    "TN": "TUN",
    "IQ": "IRQ",
    "EC": "ECU",
    "PE": "PER",
    "GH": "GHA",
    "VN": "VNM",
    "CO": "COL",
    "MY": "MYS",
    "CL": "CHL",
    "PL": "POL",
    "RO": "ROU",
}
ISO3_TO_ISO2 = {v: k for k, v in ISO2_TO_ISO3.items()}


def country_name_from_iso2(code):
    if code == "EU":
        return "European Union"
    if code in COUNTRY_NAMES:
        return COUNTRY_NAMES[code]
    country = pycountry.countries.get(alpha_2=code)
    return country.name if country else code


REGIONAL_CODES = [COUNTRY_CODE] + PEER_GROUPS["Regional"]
GLOBAL_LABEL = "Global"

In [2]:
FONT_SIZE = 11
CHAR_W = FONT_SIZE * 0.62
LINE_H = FONT_SIZE * 1.35
PAD = 6
SAFETY_PX = 4
INSIDE_LABEL_THRESHOLD = 0.20


def _wrap_label(text, value, dx, dy):
    avail_w = dx - 2 * PAD - SAFETY_PX
    avail_h = dy - 2 * PAD
    if avail_w <= 0 or avail_h <= 0:
        return None
    max_chars = int(avail_w // CHAR_W)
    max_lines = int(avail_h // LINE_H)
    if max_chars < 3 or max_lines < 1:
        return None
    value_str = f"{value:.1f}%"
    if len(value_str) > max_chars:
        return None
    words = text.split()
    if any(len(w) > max_chars for w in words):
        return None
    lines = []
    current = ""
    for word in words:
        candidate = f"{current} {word}".strip()
        if len(candidate) <= max_chars:
            current = candidate
        else:
            lines.append(current)
            current = word
            if len(lines) >= max_lines - 1:
                break
    if current and len(lines) < max_lines - 1:
        lines.append(current)
    if not lines:
        return None
    lines.append(value_str)
    return "\n".join(lines)


def make_bar(
    data,
    label_col,
    value_col,
    title,
    subtitle=None,
    width=640,
    drop_not_classified=True,
    top_n=None,
):
    d = data.copy()
    if drop_not_classified:
        d = d.loc[~d[label_col].str.contains("not_classified", case=False, na=False)]
    d = (
        d.loc[d[value_col] > 0]
        .sort_values(value_col, ascending=False)
        .reset_index(drop=True)
    )
    if top_n is not None:
        d = d.head(top_n)
    row_h = 38
    bar_size = 14
    height = max(200, row_h * len(d))
    x_max = d[value_col].max() * 1.02
    inside_cutoff = x_max * INSIDE_LABEL_THRESHOLD
    plot_df = d.assign(_zero=0.0, _value_label=d[value_col].map(lambda v: f"{v:.1f}%"))
    sort_order = plot_df[label_col].tolist()
    y_enc = alt.Y(f"{label_col}:N", sort=sort_order, title=None, axis=None)
    bars = (
        alt.Chart(plot_df)
        .mark_bar(size=bar_size)
        .encode(
            x=alt.X(
                f"{value_col}:Q",
                title="% of conversations",
                scale=alt.Scale(domain=[0, x_max], nice=False),
            ),
            y=y_enc,
            tooltip=[
                alt.Tooltip(f"{label_col}:N", title="Group"),
                alt.Tooltip(f"{value_col}:Q", title="%", format=".2f"),
            ],
        )
    )
    label_above = (
        alt.Chart(plot_df)
        .mark_text(
            align="left",
            baseline="bottom",
            fontSize=FONT_SIZE,
            color="#222",
            dy=-(bar_size // 2) - 2,
        )
        .encode(x="_zero:Q", y=y_enc, text=f"{label_col}:N")
    )
    pct_inside = (
        alt.Chart(plot_df)
        .transform_filter(f"datum['{value_col}'] >= {inside_cutoff}")
        .mark_text(
            align="right",
            baseline="middle",
            dx=-PAD,
            color="white",
            fontSize=FONT_SIZE - 1,
            fontWeight="bold",
        )
        .encode(x=f"{value_col}:Q", y=y_enc, text="_value_label:N")
    )
    pct_outside = (
        alt.Chart(plot_df)
        .transform_filter(f"datum['{value_col}'] < {inside_cutoff}")
        .mark_text(
            align="left",
            baseline="middle",
            dx=PAD,
            color="#444",
            fontSize=FONT_SIZE - 1,
        )
        .encode(x=f"{value_col}:Q", y=y_enc, text="_value_label:N")
    )
    return (bars + label_above + pct_inside + pct_outside).properties(
        width=width, height=height, title=alt.Title(text=title, subtitle=subtitle or "")
    )


def make_stacked_share(
    long_df,
    segment_col,
    segment_order,
    sort_by_segment,
    title,
    subtitle,
    palette,
    width=560,
    row_h=36,
    pin_last=None,
):
    sortable = long_df.loc[long_df["country_name"] != pin_last] if pin_last else long_df
    ordered = (
        sortable.loc[sortable[segment_col] == sort_by_segment]
        .sort_values("share", ascending=False)["country_name"]
        .tolist()
    )
    if pin_last and pin_last in long_df["country_name"].values:
        ordered.append(pin_last)
    color_enc = alt.Color(
        f"{segment_col}:N",
        scale=alt.Scale(domain=segment_order, range=palette),
        legend=alt.Legend(title=None),
    )
    y_enc = alt.Y(
        "country_name:N",
        sort=ordered,
        title=None,
        axis=alt.Axis(grid=False, ticks=False, domain=False, labelFontSize=FONT_SIZE),
    )
    order_enc = alt.Order(f"{segment_col}:N", sort="ascending")
    bars = (
        alt.Chart(long_df)
        .mark_bar(size=22)
        .encode(
            x=alt.X(
                "share:Q",
                stack="normalize",
                title=None,
                axis=alt.Axis(format="%", values=[0, 0.25, 0.5, 0.75, 1.0]),
            ),
            y=y_enc,
            color=color_enc,
            order=order_enc,
            tooltip=[
                alt.Tooltip("country_name:N", title="Country"),
                alt.Tooltip(f"{segment_col}:N", title="Segment"),
                alt.Tooltip("share:Q", title="Share", format=".1f"),
            ],
        )
    )
    labels = (
        alt.Chart(
            long_df.assign(_label=lambda d: d["share"].map(lambda v: f"{v:.0f}%"))
        )
        .transform_filter("datum.share >= 7")
        .mark_text(color="white", fontSize=FONT_SIZE - 1, fontWeight="bold")
        .encode(
            x=alt.X("share:Q", stack="normalize", bandPosition=0.5),
            y=y_enc,
            detail=f"{segment_col}:N",
            order=order_enc,
            text="_label:N",
        )
    )
    return (bars + labels).properties(
        width=width,
        height=row_h * long_df["country_name"].nunique(),
        title=alt.Title(text=title, subtitle=subtitle),
    )


def make_treemap(
    data: pd.DataFrame,
    label_col: str,
    value_col: str,
    title: str,
    subtitle: str | None = None,
    width: int = 720,
    height: int = 480,
    drop_not_classified: bool = True,
) -> alt.Chart:
    """Render a treemap from a (label, value) DataFrame using squarify + Altair.

    Labels are word-wrapped to fit each rectangle; rectangles too small to hold
    a label safely keep their tooltip but show no text.
    """
    d = data.copy()
    if drop_not_classified:
        d = d.loc[
            lambda df: (
                ~df[label_col].str.contains("not_classified", case=False, na=False)
            )
        ]
    d = (
        d.loc[lambda df: df[value_col] > 0]
        .sort_values(value_col, ascending=False)
        .reset_index(drop=True)
    )

    norm_values = squarify.normalize_sizes(d[value_col].tolist(), width, height)
    rects = squarify.squarify(norm_values, 0, 0, width, height)
    coords = pd.DataFrame(rects).assign(
        x2=lambda df: df["x"] + df["dx"], y2=lambda df: df["y"] + df["dy"]
    )
    plot_df = pd.concat([d, coords], axis=1)
    plot_df["label"] = plot_df.apply(
        lambda r: _wrap_label(str(r[label_col]), r[value_col], r["dx"], r["dy"]),
        axis=1,
    )

    base = alt.Chart(plot_df).encode(
        x=alt.X("x:Q", axis=None, scale=alt.Scale(domain=[0, width])),
        x2="x2:Q",
        y=alt.Y("y:Q", axis=None, scale=alt.Scale(domain=[0, height], reverse=True)),
        y2="y2:Q",
    )
    rects_layer = base.mark_rect(stroke="white", strokeWidth=2).encode(
        color=alt.Color(
            f"{label_col}:N",
            legend=None,
            scale=alt.Scale(range=attaviz.CATEGORICAL_TEXT),
        ),
        tooltip=[
            alt.Tooltip(f"{label_col}:N", title="Group"),
            alt.Tooltip(f"{value_col}:Q", title="% of conversations", format=".2f"),
        ],
    )
    text_layer = (
        base.transform_filter("datum.label != null")
        .mark_text(
            align="left",
            baseline="top",
            dx=PAD,
            dy=PAD,
            fontSize=FONT_SIZE,
            lineBreak="\n",
            color="white",
        )
        .encode(x="x:Q", y="y:Q", text="label:N")
    )
    if subtitle is None:
        subtitle = ""
    return (rects_layer + text_layer).properties(
        width=width, height=height, title=alt.Title(text=title, subtitle=subtitle)
    )

In [3]:
df = pd.read_csv(
    ANTHROPIC_DIR
    / "release_2026_03_24/data/aei_raw_claude_ai_2026-02-05_to_2026-02-12.csv"
)
dza = df.copy().query("geo_id == @COUNTRY_CODE")

Algeria has seen the highest jump in internet connectivity in the MENAAP Region since 2019[^1], with greater internet speeds, and improved connectivity. This improvement in infrastructure came at an opportune moment for Algerians to access and harness the power of Artificial Intelligence (AI) through usage of Large Language Models (LLMs) as part of their work.

As of March 2026, Algeria ranks 55<sup>th</sup> in the world for share of per-capita ChatGPT users[^2], and 52<sup>nd</sup> in the world for Claude usage[^3]. 38.6% of the Claude usage is from Algiers, followed by 15.9% from Blida and 12.5% from Oran. This is in line with the alternate finding that urban populations are more exposed to AI {cite:p}`oecd2024jobcreation`. 33% of Algerian Claude users use it for work, which is much lesser than the global average and its regional peers. Meanwhile, Claude usage for coursework is 2.5 times the global average, and among the highest in its regional peers, indicating that AI could change skill development and education in Algeria.

In [4]:
USE_CASE_SEGMENTS = ["personal", "work", "coursework"]

_country_use_case = df.loc[
    lambda df: (
        (df["geography"] == "country")
        & (df["facet"] == "use_case")
        & (df["variable"] == "use_case_pct")
        & (df["geo_id"].isin(REGIONAL_CODES))
        & (df["cluster_name"].isin(USE_CASE_SEGMENTS))
    )
].filter(["geo_id", "cluster_name", "value"])

_global_use_case = (
    df.loc[
        lambda df: (
            (df["geography"] == "global")
            & (df["facet"] == "use_case")
            & (df["variable"] == "use_case_pct")
            & (df["cluster_name"].isin(USE_CASE_SEGMENTS))
        )
    ]
    .filter(["cluster_name", "value"])
    .assign(geo_id="GLOBAL")
)

use_case_regional = (
    pd.concat([_country_use_case, _global_use_case], ignore_index=True)
    .assign(
        share=lambda df: df.groupby("geo_id")["value"].transform(
            lambda s: s / s.sum() * 100
        ),
        country_name=lambda df: df["geo_id"].map(
            {**COUNTRY_NAMES, "GLOBAL": GLOBAL_LABEL}
        ),
        use_case=lambda df: df["cluster_name"].str.capitalize(),
    )
    .filter(["country_name", "use_case", "share"])
)

use_case_chart = make_stacked_share(
    use_case_regional,
    segment_col="use_case",
    segment_order=["Personal", "Work", "Coursework"],
    sort_by_segment="Personal",
    title="Personal vs work use of Claude, regional peers vs global",
    subtitle="Share of classified conversations by use case category",
    palette=["#0071BC", "#F2A900", "#8A969F"],
    pin_last=GLOBAL_LABEL,
)
attaviz.add_caption(use_case_chart, "Source: Anthropic Economic Index data")

alt.VConcatChart(...)

In [5]:
wilaya = (
    df.loc[
        (df["geography"] == "country-state")
        & (df["geo_id"].str.startswith("DZ-"))
        & (df["facet"] == "country-state")
        & (df["variable"] == "usage_pct")
        & (df["geo_id"] != "DZ-not_classified")
    ]
    .assign(
        wilaya=lambda d: d["geo_id"].map(
            lambda c: getattr(pycountry.subdivisions.get(code=c), "name", c)
        ),
        value_label=lambda d: d["value"].map(lambda v: f"{v:.1f}%"),
    )[["geo_id", "wilaya", "value", "value_label"]]
    .sort_values("value", ascending=False)
    .reset_index(drop=True)
)

_y = alt.Y("wilaya:N", sort="-x", title=None)
_wilaya_base = alt.Chart(wilaya)
_wilaya_bars = _wilaya_base.mark_bar(color=ALGERIA_COLOR).encode(
    x=alt.X("value:Q", title="% of conversations"),
    y=_y,
    tooltip=[
        alt.Tooltip("wilaya:N", title="Wilaya"),
        alt.Tooltip("value:Q", title="% share", format=".1f"),
    ],
)
_wilaya_labels = _wilaya_base.mark_text(
    align="left", baseline="middle", dx=4, color="#444", fontSize=FONT_SIZE - 1
).encode(x="value:Q", y=_y, text="value_label:N")

wilaya_bar = (_wilaya_bars + _wilaya_labels).properties(
    width=560,
    height=320,
    title=alt.Title(
        text=f"Subnational Claude Usage in {COUNTRY_NAME}",
        subtitle="Share of Algeria's Claude.ai conversations by wilaya",
    ),
)
attaviz.add_caption(wilaya_bar, "Source: Anthropic Economic Index data")

alt.VConcatChart(...)

Computer and mathematical occupations account for the largest share of messages mapped to an occupational classification. This includes but is not limited to occupations such as Data Science, Statistics, Web development and Software Development. When the messages are classified based on topic, it is seen that 19% of the messages are for assisting with software development and debugging. Both these findings indicate that Algerians are either skilling up in these areas of software development or using it more as part of their jobs.

In [6]:
def request_df(level):
    return (
        dza.loc[
            lambda df: (
                (df["facet"] == "request")
                & (df["variable"] == "request_pct")
                & (df["level"] == level)
            )
        ]
        .filter(["cluster_name", "value"])
        .copy()
        .rename(columns={"value": "pct"})
        .sort_values("pct", ascending=False)
        .reset_index(drop=True)
    )


req_l2 = request_df(2)

In [7]:
req_l2_treemap = make_treemap(
    req_l2,
    label_col="cluster_name",
    value_col="pct",
    title="Intent Based Classification of Claude Usage",
)
attaviz.add_caption(req_l2_treemap, "Source: Anthropic Economic Index data")

alt.VConcatChart(...)

In [8]:
l2_bar_chart = make_bar(
    req_l2,
    label_col="cluster_name",
    value_col="pct",
    title=f"Claude Usage by Request Category in {COUNTRY_NAME}",
    subtitle="Request category derived from messages",
    top_n=10,
)
attaviz.add_caption(l2_bar_chart, "Source: Anthropic Economic Index data")

alt.VConcatChart(...)

## Underutilized, but rising software development skills in Algeria

This use of AI for coding in Algeria is complemented with an increase in developers as seen on the Open-Source Software Platform GitHub. The number of developers increased from about 330 to 1,800 per 100,000 working-age people--with a clear inflection point seen with the launch of free access AI models through GitHub copilot ({numref}`fig-developers`). A similar increase is seen across Sub-Saharan Africa, MENAAP and South Asia Region (reference). Algeria currently has more developers than Nigeria and as those in Kenya, but lesser than all its structural, regional and aspirational peers.

As of 2025, LinkedIn data suggests that the disruptive tech skill penetration in Algeria is lower than its regional peers ({numref}`fig-linkedin`). This includes skills such as Artificial Intelligence, robotics, and other skills that are meant to disrupt the labor markets in the coming years. This gap in Algeria’s skill could be bridged, in some part by the rising developer trend, and by using competence in software to its advantage.

In [9]:
GH_PEER_GROUPS = {
    "Developers": ["EG", "NG", "KE"],
    "Regional": ["EG", "JO", "TN", "IQ"],
    "Aspirational": ["MY", "CL", "PL", "RO"],
}
GH_PEER_CODES = [COUNTRY_CODE] + sorted(
    {code for codes in GH_PEER_GROUPS.values() for code in codes}
)


def _iso3_to_iso2(code):
    country = pycountry.countries.get(alpha_3=code)
    return country.alpha_2 if country else None


def load_working_age_population():
    """Working-age population (SP.POP.1564.TO) for every economy, fetched once."""
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    url = (
        "https://api.worldbank.org/v2/country/all"
        "/indicator/SP.POP.1564.TO?format=json&per_page=20000"
    )
    with urlopen(url, context=ctx) as response:
        payload = json.load(response)
    return (
        pd.DataFrame(payload[1])
        .loc[:, ["countryiso3code", "date", "value"]]
        .rename(
            columns={
                "countryiso3code": "iso3_code",
                "date": "year",
                "value": "working_age_population",
            }
        )
        .dropna(subset=["working_age_population"])
        .assign(
            iso2_code=lambda d: d["iso3_code"].map(_iso3_to_iso2),
            year=lambda d: d["year"].astype(int),
            working_age_population=lambda d: d["working_age_population"].astype(float),
        )
        .dropna(subset=["iso2_code"])
        .sort_values(["iso2_code", "year"])
    )


def attach_latest_population(github_df, population_df):
    pieces = []
    for iso2_code, country_df in github_df.sort_values("year").groupby(
        "iso2_code", sort=False
    ):
        pop_country = population_df.loc[
            population_df["iso2_code"] == iso2_code
        ].sort_values("year")
        if pop_country.empty:
            continue
        pieces.append(
            pd.merge_asof(
                country_df.sort_values("year"),
                pop_country[["year", "working_age_population"]],
                on="year",
                direction="backward",
            )
        )
    return pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame()


def load_github_metric(metric, population_df, peer_codes):
    raw = (
        pd.read_csv(GITHUB_DIR / f"{metric}.csv")
        .dropna(subset=["iso2_code"])
        .loc[lambda d: d["iso2_code"] != "EU"]
        .assign(
            iso2_code=lambda d: d["iso2_code"].astype(str),
            quarter_start=lambda d: pd.PeriodIndex(
                d["year"].astype(str) + "Q" + d["quarter"].astype(str), freq="Q"
            ).to_timestamp(),
        )
        .sort_values(["iso2_code", "year", "quarter"])
    )
    per_100k = f"{metric}_per_100k"
    return (
        attach_latest_population(raw, population_df)
        .assign(
            **{per_100k: lambda d: d[metric] / d["working_age_population"] * 100_000}
        )
        .dropna(subset=[per_100k])
        .loc[lambda d: d["iso2_code"].isin(peer_codes)]
        .assign(
            country_name=lambda d: (
                d["iso2_code"]
                .map(COUNTRY_NAMES)
                .fillna(d["iso2_code"].map(country_name_from_iso2))
            )
        )
        .sort_values(["iso2_code", "year", "quarter"])
    )


working_age_population = load_working_age_population()
developers = load_github_metric("developers", working_age_population, GH_PEER_CODES)
git_pushes = load_github_metric("git_pushes", working_age_population, GH_PEER_CODES)
clusters = pd.read_csv(GITHUB_DIR / "language_to_cluster_mapping.csv")

# Surface silent drops: the developer trend needs Kenya and Nigeria as lines.
_missing = {"KE", "NG"} - set(developers["iso2_code"])
assert not _missing, f"Reference countries dropped after population merge: {_missing}"

In [10]:
ALL_TREND_LABEL_CODES = ["DZ", "PL", "TN", "NG"]
ALL_TREND_GROUP_ORDER = [
    "Developers peers",
    "Kenya",
    "Regional peers",
    "Aspirational peers",
    "Algeria",
]
ALL_TREND_GROUP_COLORS = ["#4EC2C0", "#0C7C68", "#FF9800", "#664AB6", ALGERIA_COLOR]


def make_all_peer_trend(data, raw_col, metric_label, width=580, height=380):
    value_col = f"{raw_col}_per_100k"
    y_axis_title = f"{metric_label}/100k"
    title = f"GitHub {metric_label} per 100k Working-Age Population"
    subtitle = "Algeria and all peer countries, 2020 Q1-2025 Q4"

    trend_peer_groups = ["Developers", "Regional", "Aspirational"]
    trend_peer_codes = [c for g in trend_peer_groups for c in GH_PEER_GROUPS.get(g, [])]
    trend_codes = [COUNTRY_CODE] + trend_peer_codes
    group_of = {
        c: f"{g} peers" for g in trend_peer_groups for c in GH_PEER_GROUPS.get(g, [])
    }
    group_of[COUNTRY_CODE] = "Algeria"
    label_codes = (
        set(ALL_TREND_LABEL_CODES) | set(GH_PEER_GROUPS.get("Developers", []))
    ) - {"KE"}

    d = data.loc[data["iso2_code"].isin(trend_codes)].copy()
    d["peer_group"] = d["iso2_code"].map(group_of)
    d.loc[d["iso2_code"] == "KE", "peer_group"] = "Kenya"
    d["country_name"] = (
        d["iso2_code"]
        .map(COUNTRY_NAMES)
        .fillna(d["iso2_code"].map(country_name_from_iso2))
    )
    latest_quarter = d["quarter_start"].max()
    labels = d.loc[
        (d["quarter_start"] == latest_quarter) & d["iso2_code"].isin(label_codes)
    ].copy()

    color = alt.Color(
        "peer_group:N",
        title=None,
        scale=alt.Scale(domain=ALL_TREND_GROUP_ORDER, range=ALL_TREND_GROUP_COLORS),
        legend=alt.Legend(orient="bottom", direction="horizontal", columns=3),
    )
    base = alt.Chart(d).encode(
        x=alt.X("quarter_start:T", title=None),
        y=alt.Y(
            f"{value_col}:Q",
            title=y_axis_title,
            axis=alt.Axis(format="~s"),
            scale=alt.Scale(zero=True),
        ),
        color=color,
        detail="country_name:N",
    )
    tooltip = [
        alt.Tooltip("country_name:N", title="Country"),
        alt.Tooltip("peer_group:N", title="Peer group"),
        alt.Tooltip("year:O", title="Year"),
        alt.Tooltip("quarter:O", title="Quarter"),
        alt.Tooltip(f"{value_col}:Q", title=y_axis_title, format=",.0f"),
        alt.Tooltip(
            f"{raw_col}:Q", title=raw_col.replace("_", " ").title(), format=","
        ),
        alt.Tooltip(
            "working_age_population:Q", title="Working-age population", format=",.0f"
        ),
    ]
    peer_lines = (
        base.transform_filter(alt.datum.iso2_code != COUNTRY_CODE)
        .mark_line()
        .encode(tooltip=tooltip)
    )
    algeria_line = (
        base.transform_filter(alt.datum.iso2_code == COUNTRY_CODE)
        .mark_line()
        .encode(tooltip=tooltip)
    )
    latest_labels = (
        alt.Chart(labels)
        .mark_text(align="left", baseline="middle", dx=6, fontSize=11)
        .encode(
            x="quarter_start:T",
            y=f"{value_col}:Q",
            text="country_name:N",
            color=alt.Color(
                "peer_group:N",
                scale=alt.Scale(
                    domain=ALL_TREND_GROUP_ORDER, range=ALL_TREND_GROUP_COLORS
                ),
                legend=None,
            ),
        )
    )
    copilot = pd.DataFrame(
        {
            "quarter_start": [pd.Timestamp("2024-12-15")],
            "label": ["Launch of free access to GitHub Copilot"],
        }
    )
    annotation_rule = (
        alt.Chart(copilot)
        .mark_rule(color="#8A969F", strokeDash=[8, 4], strokeWidth=1.2)
        .encode(x="quarter_start:T")
    )
    annotation_text = (
        alt.Chart(copilot)
        .mark_text(
            align="left", baseline="top", dx=6, dy=6, color="#111111", fontSize=11
        )
        .encode(x="quarter_start:T", y=alt.value(0), text="label:N")
    )
    return alt.layer(
        peer_lines, algeria_line, annotation_rule, annotation_text, latest_labels
    ).properties(
        width=width, height=height, title=alt.Title(text=title, subtitle=subtitle)
    )


all_developers_trend = make_all_peer_trend(
    developers, raw_col="developers", metric_label="Developers"
)
attaviz.add_caption(
    all_developers_trend,
    "Source: GitHub Innovation Graph data; World Bank SP.POP.1564.TO",
)

alt.VConcatChart(...)

In [11]:
# LinkedIn relative penetration of Disruptive Tech Skills, Algeria vs comparators.
# Morocco is excluded from the comparator set (project decision).
LINKEDIN_FILE = DATA_DIR / "LinkedIn" / "Skill Genome and Skills Pen 2025.xlsx"
LINKEDIN_COMPARATORS = [
    "Algeria",
    "Chile",
    "Poland",
    "Malaysia",
    "Turkiye",
    "Peru",
    "Nigeria",
    "Egypt",
    "Colombia",
    "Tunisia",
    "Romania",
    "Ghana",
    "Iraq",
    "Ecuador",
    "Vietnam",
    "Jordan",
    "Kenya",
]

_skill_pen = (
    pd.read_excel(LINKEDIN_FILE, sheet_name="3A - SPP Ctry", header=5)
    .dropna(subset=["Country", "Skill", "Average", "Global", "Relative"])
    .assign(Country=lambda d: d["Country"].str.strip())
)
tech_skills = (
    _skill_pen.loc[
        lambda df: (
            (df["Skill"] == "Disruptive Tech Skills")
            & df["Country"].isin(LINKEDIN_COMPARATORS)
        )
    ][["Country", "Relative"]]
    .sort_values("Relative", ascending=False)
    .reset_index(drop=True)
    .assign(relative_label=lambda d: d["Relative"].map(lambda v: f"{v:.2f}"))
)

_ls_y = alt.Y("Country:N", sort=tech_skills["Country"].tolist(), title=None)
_ls_base = alt.Chart(tech_skills)
_ls_bars = _ls_base.mark_bar().encode(
    x=alt.X("Relative:Q", title="Relative penetration"),
    y=_ls_y,
    color=alt.condition(
        alt.datum.Country == COUNTRY_NAME,
        alt.value(ALGERIA_COLOR),
        alt.value("#8A969F"),
    ),
    tooltip=[alt.Tooltip("Country:N"), alt.Tooltip("Relative:Q", format=".2f")],
)
_ls_labels = _ls_base.mark_text(
    align="left", baseline="middle", dx=4, color="#444", fontSize=FONT_SIZE - 1
).encode(x="Relative:Q", y=_ls_y, text="relative_label:N")

linkedin_skills_chart = (_ls_bars + _ls_labels).properties(
    width=560,
    height=26 * len(tech_skills),
    title=alt.Title(
        text="Relative Penetration of Disruptive Tech Skills",
        subtitle="Algeria and comparator countries",
    ),
)
attaviz.add_caption(
    linkedin_skills_chart, "Source: LinkedIn Skill Genome and Skills Penetration 2025"
)

alt.VConcatChart(...)

Egypt, Morocco, Nigeria, Ghana, Kenya are all projected to add millions of developers in the coming years {cite:p}`github2025octoverse`. Developers in Algeria use similar software languages compared to what is used in these countries, with web markup and styling languages being the most commonly prevalent. This skillset allows these countries to become an outsourcing destination for software; a position Algeria can set itself up for.

GitHub developers in Algeria have long been collaborating with other counterparts in other countries, dominated by those in the European Union and United States ({numref}`fig-flows`). Such cross-border software and technical collaboration are considered a form of digital trade {cite:p}`juhasz2026software`. The AI revolution presents Algeria with a strategic opportunity to expand this digital trade, adding a new and high-value sectoral dimension to its economy.

In [12]:
language_country_codes = [COUNTRY_CODE, "EG", "NG"]
language_country_order = [COUNTRY_NAMES[c] for c in language_country_codes]
language_country_colors = [ALGERIA_COLOR, "#FF9800", "#4EC2C0"]

languages = (
    pd.read_csv(GITHUB_DIR / "languages.csv")
    .loc[lambda d: d["iso2_code"].isin(language_country_codes)]
    .assign(
        quarter_start=lambda d: pd.PeriodIndex(
            d["year"].astype(str) + "Q" + d["quarter"].astype(str), freq="Q"
        ).to_timestamp(),
        country_name=lambda d: d["iso2_code"].map(COUNTRY_NAMES),
    )
)
languages["cluster"] = languages["language"].map(
    clusters.set_index("Language")["Cluster Name"]
)
languages = (
    languages.dropna(subset=["cluster"])
    .groupby(
        ["iso2_code", "country_name", "cluster", "quarter_start", "year", "quarter"],
        as_index=False,
    )["num_pushers"]
    .sum()
)
languages = attach_latest_population(languages, working_age_population).assign(
    num_pushers_per_100k=lambda d: (
        d["num_pushers"] / d["working_age_population"] * 100_000
    )
)
latest_language_quarter = languages["quarter_start"].max()
latest_language_rows = languages.loc[
    languages["quarter_start"] == latest_language_quarter
].copy()
latest_language_label = (
    f"{latest_language_rows['year'].iloc[0]} Q{latest_language_rows['quarter'].iloc[0]}"
)

top_cluster_names = (
    latest_language_rows.sort_values(
        ["country_name", "num_pushers_per_100k"], ascending=[True, False]
    )
    .groupby("iso2_code", as_index=False, group_keys=False)
    .head(10)["cluster"]
    .drop_duplicates()
    .tolist()
)
language_comparison = latest_language_rows.loc[
    lambda d: d["cluster"].isin(top_cluster_names)
].copy()
cluster_sort = (
    language_comparison.groupby("cluster", as_index=False)["num_pushers_per_100k"]
    .sum()
    .sort_values("num_pushers_per_100k", ascending=False)["cluster"]
    .tolist()
)

languages_chart = (
    alt.Chart(language_comparison)
    .mark_bar(size=10)
    .encode(
        x=alt.X(
            "num_pushers_per_100k:Q",
            title="Pushers per 100k working-age population",
            axis=alt.Axis(format="~s"),
        ),
        y=alt.Y(
            "cluster:N", sort=cluster_sort, title=None, axis=alt.Axis(labelLimit=0)
        ),
        color=alt.Color(
            "country_name:N",
            title=None,
            scale=alt.Scale(
                domain=language_country_order, range=language_country_colors
            ),
            legend=alt.Legend(orient="bottom", direction="horizontal"),
        ),
        yOffset=alt.YOffset("country_name:N", sort=language_country_order),
        tooltip=[
            alt.Tooltip("country_name:N", title="Country"),
            alt.Tooltip("cluster:N", title="Cluster"),
            alt.Tooltip(
                "num_pushers_per_100k:Q", title="Pushers per 100k", format=",.1f"
            ),
            alt.Tooltip("num_pushers:Q", title="Pushers", format=","),
        ],
    )
    .properties(
        width=620,
        height=max(320, 34 * len(cluster_sort)),
        title=alt.Title(
            text="Top GitHub Language Clusters in Algeria, Egypt and Nigeria",
            subtitle=(
                "Union of each country's top 10 clusters by pushers per 100k "
                f"working-age population, latest available quarter: {latest_language_label}"
            ),
            fontSize=14,
            subtitleFontSize=11,
            offset=10,
        ),
    )
)
attaviz.add_caption(languages_chart, "Source: GitHub Innovation Graph languages data")

alt.VConcatChart(...)

In [13]:
flows_collaborators = (
    pd.read_csv(GITHUB_DIR / "economy_collaborators.csv")
    .loc[lambda d: (d["source"] == COUNTRY_CODE) | (d["destination"] == COUNTRY_CODE)]
    .assign(
        quarter_start=lambda d: pd.PeriodIndex(
            d["year"].astype(str) + "Q" + d["quarter"].astype(str), freq="Q"
        ).to_timestamp()
    )
)

EUROPE_ISO2 = {
    "AL",
    "AD",
    "AT",
    "BY",
    "BE",
    "BA",
    "BG",
    "HR",
    "CY",
    "CZ",
    "DK",
    "EE",
    "FI",
    "FR",
    "DE",
    "GR",
    "HU",
    "IS",
    "IE",
    "IT",
    "XK",
    "LV",
    "LI",
    "LT",
    "LU",
    "MT",
    "MD",
    "MC",
    "ME",
    "NL",
    "MK",
    "NO",
    "PL",
    "PT",
    "RO",
    "RU",
    "SM",
    "RS",
    "SK",
    "SI",
    "ES",
    "SE",
    "CH",
    "UA",
    "GB",
    "VA",
}
_excluded_partners = EUROPE_ISO2 - {"EU"}
_required_partners = ["US", "EU"]


def prepare_algeria_collaborator_flow(flow):
    if flow == "Outbound":
        base = flows_collaborators.loc[lambda d: d["source"] == COUNTRY_CODE].copy()
        partner_col = "destination"
    else:
        base = flows_collaborators.loc[
            lambda d: d["destination"] == COUNTRY_CODE
        ].copy()
        partner_col = "source"
    base = (
        base.loc[lambda d: ~d[partner_col].isin(_excluded_partners)]
        .rename(columns={partner_col: "partner"})
        .groupby(["quarter_start", "year", "quarter", "partner"], as_index=False)[
            "weight"
        ]
        .sum()
    )
    latest_q = base.loc[base["year"] == 2025, "quarter_start"].max()
    latest_label = (
        base.loc[base["quarter_start"] == latest_q, ["year", "quarter"]]
        .drop_duplicates()
        .assign(
            label=lambda d: d["year"].astype(str) + " Q" + d["quarter"].astype(str)
        )["label"]
        .iloc[0]
    )
    top_other = (
        base.loc[lambda d: d["quarter_start"] == latest_q]
        .groupby("partner", as_index=False)["weight"]
        .sum()
        .loc[lambda d: ~d["partner"].isin(_required_partners)]
        .sort_values("weight", ascending=False)
        .head(3)["partner"]
        .tolist()
    )
    selected = _required_partners + top_other
    flow_data = base.loc[lambda d: d["partner"].isin(selected)].assign(
        partner_name=lambda d: d["partner"].map(country_name_from_iso2),
        partner_order=lambda d: d["partner"].map(
            {c: i for i, c in enumerate(selected)}
        ),
        flow=flow,
    )
    order = (
        flow_data.drop_duplicates("partner")
        .sort_values("partner_order")["partner_name"]
        .tolist()
    )
    return flow_data, order, latest_label


def make_collaborator_area_panel(flow, title, width=540, height=300):
    flow_data, order, latest_label = prepare_algeria_collaborator_flow(flow)
    return (
        alt.Chart(flow_data)
        .mark_area(opacity=0.82)
        .encode(
            x=alt.X("quarter_start:T", title=None),
            y=alt.Y(
                "weight:Q",
                title="Collaboration weight",
                stack="zero",
                axis=alt.Axis(format="~s"),
            ),
            color=alt.Color(
                "partner_name:N",
                title="Partner",
                scale=alt.Scale(domain=order),
                legend=alt.Legend(orient="bottom", columns=3),
            ),
            order=alt.Order("partner_order:Q", sort="ascending"),
            tooltip=[
                alt.Tooltip("partner_name:N", title="Partner"),
                alt.Tooltip("year:O", title="Year"),
                alt.Tooltip("quarter:O", title="Quarter"),
                alt.Tooltip("weight:Q", title="Collaboration weight", format=","),
            ],
        )
        .properties(
            width=width,
            height=height,
            title=alt.Title(
                text=title,
                subtitle=f"US, EU, and top three non-European partners in {latest_label}",
            ),
        )
    )


outbound_flows_chart = make_collaborator_area_panel(
    "Outbound", "Outbound GitHub collaborator flows from Algeria"
)
attaviz.add_caption(
    outbound_flows_chart, "Source: GitHub Innovation Graph economy collaborators data"
)

alt.VConcatChart(...)

## References

```{bibliography}
:filter: docname in docnames or key == "chatterji2025chatgpt"
```

[^1]: World Bank analysis using Ookla Speedtest data from Ookla for Good https://datapartnership.org/algeria-economic-monitoring/notebooks/ookla-analysis/ookla-visualization-comparison.html

[^2]: OpenAI Signals Report, published on May 11, 2026. This rank is created based on a sample of messages between July 2024 and March 2026.

[^3]: Rank created based on the share of messages from Algeria, released in the Anthropic Economic Index Report 2026 {cite:p}`massenkoff2026anthropic`. The data is messages from 5th to 12<sup>th</sup> February 2026.